# Build EIA State-Year Controls, 2001-2024

This notebook cleans U.S. state-year economic and electricity-system controls used in the EIA coal-generation mechanism analysis.

Main output: `Data/temp/eia_state_year_controls_2001_2024.csv`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    while current.name != "Data_center_and_fossil_energy_Replication" and current.parent != current:
        current = current.parent
    if current.name != "Data_center_and_fossil_energy_Replication":
        raise RuntimeError(
            "Project root 'Data_center_and_fossil_energy_Replication' could not be located. "
            "Run this notebook from inside the replication repository."
        )
    return current


BASE_PATH = find_project_root()
RAW = BASE_PATH / "Data" / "raw" / "eia_state_controls"
TEMP = BASE_PATH / "Data" / "temp"

STATE_CONTROLS_CSV = TEMP / "eia_state_year_controls_2001_2024.csv"

YEAR_MIN = 2001
YEAR_MAX = 2024
GROWTH_BASE_YEAR = YEAR_MIN - 1

STATE_FIPS_TO_ABBR = {
    "01": "AL",
    "02": "AK",
    "04": "AZ",
    "05": "AR",
    "06": "CA",
    "08": "CO",
    "09": "CT",
    "10": "DE",
    "11": "DC",
    "12": "FL",
    "13": "GA",
    "15": "HI",
    "16": "ID",
    "17": "IL",
    "18": "IN",
    "19": "IA",
    "20": "KS",
    "21": "KY",
    "22": "LA",
    "23": "ME",
    "24": "MD",
    "25": "MA",
    "26": "MI",
    "27": "MN",
    "28": "MS",
    "29": "MO",
    "30": "MT",
    "31": "NE",
    "32": "NV",
    "33": "NH",
    "34": "NJ",
    "35": "NM",
    "36": "NY",
    "37": "NC",
    "38": "ND",
    "39": "OH",
    "40": "OK",
    "41": "OR",
    "42": "PA",
    "44": "RI",
    "45": "SC",
    "46": "SD",
    "47": "TN",
    "48": "TX",
    "49": "UT",
    "50": "VT",
    "51": "VA",
    "53": "WA",
    "54": "WV",
    "55": "WI",
    "56": "WY",
}

RENEWABLE_GENERATION_SOURCES = {
    "Hydroelectric Conventional",
    "Wind",
    "Solar Thermal and Photovoltaic",
    "Wood and Wood Derived Fuels",
    "Other Biomass",
    "Geothermal",
}

RENEWABLE_CAPACITY_SOURCES = {
    "Hydroelectric",
    "Wind",
    "Solar Thermal and Photovoltaic",
    "Wood and Wood Derived Fuels",
    "Other Biomass",
    "Geothermal",
}


def clean_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).replace(
            {"(NA)": np.nan, "NA": np.nan, "nan": np.nan}
        ),
        errors="coerce",
    )


def keep_analysis_years(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["year"].between(YEAR_MIN, YEAR_MAX)].copy()


def keep_growth_base_years(df: pd.DataFrame) -> pd.DataFrame:
    return df[df["year"].between(GROWTH_BASE_YEAR, YEAR_MAX)].copy()


def first_existing_col(df: pd.DataFrame, names: list[str]) -> str:
    for name in names:
        if name in df.columns:
            return name
    raise KeyError(f"None of these columns were found: {names}")


def read_retail_sales_file(path: Path) -> pd.DataFrame:
    df = pd.read_excel(
        path,
        sheet_name="Total Electric Industry",
        header=2,
    )
    year_col = first_existing_col(df, ["Year", "YEAR"])
    total_sales_col = "Megawatthours.5" if "Megawatthours.5" in df.columns else "Megawatthours.4"
    total_price_col = "Cents/kWh.5" if "Cents/kWh.5" in df.columns else "Cents/kWh.4"
    out = pd.DataFrame(
        {
            "state": df["STATE"].astype(str).str.strip(),
            "year": clean_numeric(df[year_col]).astype("Int64"),
            "st_res_sales_mwh": clean_numeric(df["Megawatthours"]),
            "st_comm_sales_mwh": clean_numeric(df["Megawatthours.1"]),
            "st_ind_sales_mwh": clean_numeric(df["Megawatthours.2"]),
            "st_total_sales_mwh": clean_numeric(df[total_sales_col]),
            "st_res_price_cents_kwh": clean_numeric(df["Cents/kWh"]),
            "st_comm_price_cents_kwh": clean_numeric(df["Cents/kWh.1"]),
            "st_ind_price_cents_kwh": clean_numeric(df["Cents/kWh.2"]),
            "st_retail_price_cents_kwh": clean_numeric(df[total_price_col]),
        }
    )
    out = out[out["state"].str.len() == 2]
    out = out[out["year"].notna()].copy()
    out["year"] = out["year"].astype(int)
    return out


def build_retail_sales_controls() -> pd.DataFrame:
    files = [
        RAW / "eia_state_sales_revenue_price_1960_1989.xlsx",
        RAW / "eia_state_sales_revenue_price_1990_2009.xlsx",
        RAW / "eia_state_sales_revenue_price_2010_2024.xlsx",
    ]
    out = pd.concat(
        [read_retail_sales_file(path) for path in files],
        ignore_index=True,
    )
    out = (
        out.sort_values(["state", "year"])
        .drop_duplicates(["state", "year"], keep="last")
        .reset_index(drop=True)
    )
    return keep_growth_base_years(out)

def build_generation_controls() -> pd.DataFrame:
    path = RAW / "eia_state_generation_1990_2024.xls"
    df = pd.read_excel(
        path,
        sheet_name="Net_Generation_1990-2024 Final",
        header=1,
    )
    df = df[
        df["TYPE OF PRODUCER"].eq("Total Electric Power Industry")
    ].copy()
    df["year"] = clean_numeric(df["YEAR"]).astype("Int64")
    df["state"] = df["STATE"].astype(str).str.strip()
    df["generation_mwh"] = clean_numeric(df["GENERATION (Megawatthours)"])
    df = df[df["state"].str.len() == 2]
    df["year"] = df["year"].astype(int)
    df = keep_growth_base_years(df)

    def source_sum(sources: set[str], name: str) -> pd.DataFrame:
        x = df[df["ENERGY SOURCE"].isin(sources)]
        return (
            x.groupby(["state", "year"], as_index=False)["generation_mwh"]
            .sum()
            .rename(columns={"generation_mwh": name})
        )

    total = source_sum({"Total"}, "st_total_gen_mwh")
    coal = source_sum({"Coal"}, "st_coal_gen_mwh")
    gas = source_sum({"Natural Gas"}, "st_gas_gen_mwh")
    renew = source_sum(RENEWABLE_GENERATION_SOURCES, "st_ren_gen_mwh")

    out = total.merge(coal, on=["state", "year"], how="left")
    out = out.merge(gas, on=["state", "year"], how="left")
    out = out.merge(renew, on=["state", "year"], how="left")
    for col in ["st_coal_gen_mwh", "st_gas_gen_mwh", "st_ren_gen_mwh"]:
        out[col] = out[col].fillna(0)
    out["st_coal_gen_share"] = out["st_coal_gen_mwh"] / out[
        "st_total_gen_mwh"
    ].replace(0, np.nan)
    out["st_gas_gen_share"] = out["st_gas_gen_mwh"] / out[
        "st_total_gen_mwh"
    ].replace(0, np.nan)
    out["st_ren_gen_share"] = out["st_ren_gen_mwh"] / out[
        "st_total_gen_mwh"
    ].replace(0, np.nan)
    return out


def build_capacity_controls() -> pd.DataFrame:
    path = RAW / "eia_state_existing_capacity_1990_2024.xlsx"
    df = pd.read_excel(
        path,
        sheet_name="Existing Capacity",
        header=1,
    )
    df = df[
        df["Producer Type"].eq("Total Electric Power Industry")
    ].copy()
    df["year"] = clean_numeric(df["Year"]).astype("Int64")
    df["state"] = df["State Code"].astype(str).str.strip()
    df["nameplate_mw"] = clean_numeric(
        df["Nameplate Capacity (Megawatts)"]
    )
    df = df[df["state"].str.len() == 2]
    df["year"] = df["year"].astype(int)
    df = keep_growth_base_years(df)

    def source_sum(sources: set[str], name: str) -> pd.DataFrame:
        x = df[df["Fuel Source"].isin(sources)]
        return (
            x.groupby(["state", "year"], as_index=False)["nameplate_mw"]
            .sum()
            .rename(columns={"nameplate_mw": name})
        )

    total = source_sum({"All Sources"}, "st_total_cap_mw")
    coal = source_sum({"Coal"}, "st_coal_cap_mw")
    gas = source_sum({"Natural Gas"}, "st_gas_cap_mw")
    renew = source_sum(RENEWABLE_CAPACITY_SOURCES, "st_ren_cap_mw")

    out = total.merge(coal, on=["state", "year"], how="left")
    out = out.merge(gas, on=["state", "year"], how="left")
    out = out.merge(renew, on=["state", "year"], how="left")
    for col in ["st_coal_cap_mw", "st_gas_cap_mw", "st_ren_cap_mw"]:
        out[col] = out[col].fillna(0)
    out["st_coal_cap_share"] = out["st_coal_cap_mw"] / out[
        "st_total_cap_mw"
    ].replace(0, np.nan)
    out["st_gas_cap_share"] = out["st_gas_cap_mw"] / out[
        "st_total_cap_mw"
    ].replace(0, np.nan)
    out["st_ren_cap_share"] = out["st_ren_cap_mw"] / out[
        "st_total_cap_mw"
    ].replace(0, np.nan)
    return out


def build_gdp_controls() -> pd.DataFrame:
    path = RAW / "bea_sagdp_state_gdp" / "SAGDP1__ALL_AREAS_1997_2025.csv"
    df = pd.read_csv(path)
    df["line_code"] = clean_numeric(df["LineCode"])
    df = df[df["line_code"].isin([1, 3])].copy()
    df["state_fips"] = (
        df["GeoFIPS"]
        .astype(str)
        .str.strip()
        .str.replace('"', "", regex=False)
        .str.zfill(5)
    )
    df["state"] = df["state_fips"].str[:2].map(STATE_FIPS_TO_ABBR)
    df = df[df["state"].notna()]

    year_cols = [str(y) for y in range(1997, 2026) if str(y) in df.columns]
    long = df.melt(
        id_vars=["state", "line_code"],
        value_vars=year_cols,
        var_name="year",
        value_name="value",
    )
    long["year"] = long["year"].astype(int)
    long["value"] = clean_numeric(long["value"])
    wide = long.pivot_table(
        index=["state", "year"], columns="line_code", values="value"
    ).reset_index()
    wide.columns = [
        "st_real_gdp_mil"
        if c in [1, 1.0, "1", "1.0"]
        else "st_current_gdp_mil"
        if c in [3, 3.0, "3", "3.0"]
        else c
        for c in wide.columns
    ]
    wide = keep_growth_base_years(wide)
    wide["st_ln_real_gdp"] = np.log(wide["st_real_gdp_mil"])
    return wide


def add_growth_rates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["state", "year"]).copy()
    growth_specs = {
        "st_real_gdp_mil": "st_real_gdp_growth_pct",
        "st_total_sales_mwh": "st_sales_growth_pct",
        "st_total_gen_mwh": "st_gen_growth_pct",
    }
    for source, target in growth_specs.items():
        df[target] = (
            df.groupby("state")[source].pct_change(fill_method=None) * 100
        )
    return df


def build_state_controls() -> pd.DataFrame:
    controls = build_retail_sales_controls()
    for component in [
        build_generation_controls(),
        build_capacity_controls(),
        build_gdp_controls(),
    ]:
        controls = controls.merge(component, on=["state", "year"], how="outer")
    controls = add_growth_rates(controls)
    controls = keep_analysis_years(controls)
    controls = controls.sort_values(["state", "year"]).reset_index(drop=True)
    return controls


def write_outputs(controls: pd.DataFrame) -> None:
    TEMP.mkdir(parents=True, exist_ok=True)
    controls.to_csv(STATE_CONTROLS_CSV, index=False)


def main() -> None:
    print("Building state-year controls")
    controls = build_state_controls()
    print(
        f"State-year controls: {len(controls):,} rows, "
        f"{controls['state'].nunique()} states, "
        f"{controls['year'].min()}-{controls['year'].max()}"
    )

    write_outputs(controls)
    print(f"Saved controls CSV: {STATE_CONTROLS_CSV}")
    print(controls.head().to_string(index=False))


In [ ]:
main()
